# Assignment 1 — Query Expansion Retrieval

Generates multiple alternative phrasings of a user query, retrieves evidence for each in parallel, deduplicates results, and synthesises a cited answer.

In [1]:
import os, re, asyncio, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import chromadb
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI, AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "")

if NVIDIA_API_KEY:
    BASE_URL = "https://integrate.api.nvidia.com/v1"
    API_KEY  = NVIDIA_API_KEY
    MODEL    = "meta/llama-3.1-8b-instruct"
    PROVIDER = "nvidia"
elif GROQ_API_KEY:
    BASE_URL = "https://api.groq.com/openai/v1"
    API_KEY  = GROQ_API_KEY
    MODEL    = "llama-3.1-8b-instant"
    PROVIDER = "groq"
else:
    raise ValueError("Set NVIDIA_API_KEY or GROQ_API_KEY in .env")

print(f"Provider: {PROVIDER} | Model: {MODEL}")

COLLECTION_NAME = "tesla-10k-rechunked"   # use the rechunked collection from Assignment 2
TOP_K           = 5

Provider: nvidia | Model: meta/llama-3.1-8b-instruct


In [2]:
print("Loading embeddings...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"batch_size": 128},
)

chromadb_client = chromadb.PersistentClient(path="./tesla_db")
available = list(chromadb_client.list_collections())
col = COLLECTION_NAME if COLLECTION_NAME in available else "tesla-10k-2019-to-2023"
print(f"Using collection: {col}")

vectorstore = Chroma(
    collection_name=col,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db",
)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
print("Ready.")

Loading embeddings...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Using collection: tesla-10k-rechunked
Ready.


In [3]:
EXPANSION_SYSTEM = """You are a financial domain expert helping answer questions from Tesla 10-K filings.
Perform query expansion on the question below. Generate 4 to 6 alternative phrasings
covering all of these angles:
1. Financial analyst phrasing (revenue, margins, capital allocation)
2. Risk-factor phrasing (SEC disclosure language, material risks)
3. Operational phrasing (execution, supply chain, manufacturing)
4. Synonym / subtopic phrasing (related concepts, alternative terminology)

Rules:
- Output ONLY a plain list of questions, one per line
- No numbering, no bullets, no extra text
- Preserve the original intent — do not create unrelated queries"""

ANSWER_SYSTEM = """You are an assistant to a financial services firm answering questions from Tesla annual reports.
Use ONLY the provided context. Every factual claim must cite its source as
(source: <source_doc>, <fiscal_year>, <section>, chunk <chunk_id>).
If the answer is not in the context, say "I don't know"."""

sync_client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

print("Clients ready.")


Clients ready.


In [4]:
def expand_query(question: str) -> list[str]:
    resp = sync_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": EXPANSION_SYSTEM},
            {"role": "user",   "content": f"<Question>\n{question}\n</Question>"},
        ],
        temperature=0,
        max_tokens=512,
    )
    expansions = [q.strip() for q in resp.choices[0].message.content.strip().split("\n") if q.strip()]
    if question not in expansions:
        expansions.insert(0, question)
    return expansions

async def retrieve_all_parallel(expanded_queries: list[str]) -> list[tuple[str, list]]:
    """Returns list of (query_string, docs) so retrieved_by can be tracked."""
    async def fetch(q):
        loop = asyncio.get_event_loop()
        docs = await loop.run_in_executor(None, retriever.invoke, q)
        return (q, docs)
    results = await asyncio.gather(*[fetch(q) for q in expanded_queries])
    return results

def deduplicate_with_source(query_doc_pairs: list[tuple[str, list]]) -> list:
    """Deduplicate and tag each doc with the query that first retrieved it."""
    seen, unique = set(), []
    for query, docs in query_doc_pairs:
        for d in docs:
            norm = re.sub(r"\s+", " ", d.page_content).strip()[:200]
            if norm not in seen:
                seen.add(norm)
                d.metadata["retrieved_by"] = query
                unique.append(d)
    return unique

def generate_answer(question: str, context_docs: list) -> str:
    context_parts = []
    for d in context_docs:
        m = d.metadata
        context_parts.append(
            f"[{m.get('chunk_id','?')} | {m.get('source_doc','?')} | {m.get('fiscal_year','?')} | {m.get('section','?')}]\n{d.page_content}"
        )
    context = "\n\n".join(context_parts)
    resp = sync_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user",   "content": f"<Context>\n{context}\n</Context>\n\n<Question>\n{question}\n</Question>"},
        ],
        temperature=0,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

print("Functions defined.")


Functions defined.


## Run on a single query (interactive use)

In [5]:
user_query = "What was the automotive revenue in 2021?"

t0 = time.time()
expanded = expand_query(user_query)
print(f"Expanded into {len(expanded)} queries:")
for q in expanded:
    print(f"  • {q}")
all_retrieved = asyncio.run(retrieve_all_parallel(expanded))
unique_docs = deduplicate_with_source(all_retrieved)
print(f"\nTotal retrieved: {sum(len(r) for r in all_retrieved)} | Unique after dedup: {len(unique_docs)}")
answer = generate_answer(user_query, unique_docs)
print(f"\nAnswer:\n{answer}")
print(f"\nTotal time: {time.time()-t0:.1f}s")

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/salescode/.pyenv/versions/3.11.9/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a5b2f40> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/salescode/.pyenv/versions/3.11.9/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x10a5b2f40> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/salescode/.pyenv/versions/3.11.9/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_con

Expanded into 4 queries:
  • What was the automotive revenue in 2021?
  • What were the total revenues from automotive sales for the fiscal year ended December 31, 2021?
  • What were the revenues generated from the sale of automotive products and services in 2021?
  • What were the total revenues from the automotive segment for the year ended December 31, 2021?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Total retrieved: 20 | Unique after dedup: 7

Answer:
According to the provided context in [doc_1468 | tsla-20221231-gen.pdf | 2022 | ?], the total automotive revenues for 2021 were $47,232.

Total time: 4.8s


## Benchmark Questions (Q1–Q4)

In [6]:
BENCHMARK_QUESTIONS = {
    "Q1": "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.",
    "Q2": "Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims.",
    "Q3": "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations.",
    "Q4": "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?",
}

results = {}

for qid, question in BENCHMARK_QUESTIONS.items():
    print(f"\n{'='*70}")
    print(f"[{qid}] {question[:80]}...")
    print(f"{'='*70}")

    t0 = time.time()

    baseline_docs = retriever.invoke(question)

    expanded = expand_query(question)
    all_retrieved = asyncio.run(retrieve_all_parallel(expanded))
    unique_docs = deduplicate_with_source(all_retrieved)

    answer = generate_answer(question, unique_docs)
    elapsed = time.time() - t0

    print(f"Baseline chunks : {len(baseline_docs)}")
    print(f"Expanded chunks : {len(unique_docs)} (from {len(expanded)} queries)")
    print(f"Time            : {elapsed:.1f}s")
    print(f"\nAnswer:\n{answer}")

    results[qid] = {
        "question_id": qid,
        "original_query": question,
        "expanded_queries": expanded,
        "baseline_top_chunks": [
            {
                "chunk_id": d.metadata.get("chunk_id", ""),
                "section":  d.metadata.get("section", ""),
                "year":     str(d.metadata.get("fiscal_year", "")),
                "score":    None,
                "text":     d.page_content[:300],
            }
            for d in baseline_docs
        ],
        "expanded_top_chunks": [
            {
                "chunk_id":     d.metadata.get("chunk_id", ""),
                "section":      d.metadata.get("section", ""),
                "year":         str(d.metadata.get("fiscal_year", "")),
                "score":        None,
                "retrieved_by": "expanded query",
                "text":         d.page_content[:300],
            }
            for d in unique_docs
        ],
        "final_answer": answer,
        "citations": [
            {
                "chunk_id":   d.metadata.get("chunk_id", ""),
                "source_doc": d.metadata.get("source_doc", ""),
                "section":    d.metadata.get("section", ""),
                "year":       str(d.metadata.get("fiscal_year", "")),
            }
            for d in unique_docs
        ],
        "retrieval_improvement_analysis": (
            f"Expanded retrieval returned {len(unique_docs)} unique chunks vs "
            f"{len(baseline_docs)} baseline chunks using {len(expanded)} query variants."
        ),
    }

print("\nAll benchmark questions complete.")



[Q1] Does Tesla's growth story appear more constrained by external supply risk or int...
Baseline chunks : 5
Expanded chunks : 9 (from 4 queries)
Time            : 9.7s

Answer:
Based on the provided context, Tesla's growth story appears to be more constrained by internal execution and cost structure rather than external supply risk.

Evidence from the Risk Factors section suggests that Tesla is vulnerable to various external risks, including natural disasters, wars, health epidemics, and other events outside of their control (Source: <doc_1 | tsla-10k_20191231-gen_0.pdf | 2019 | ?>, Risk Factors, Item 1A). However, these risks are not explicitly stated as constraints on Tesla's growth story.

In contrast, the MD&A section highlights several internal execution and cost structure risks that could impact Tesla's growth. For example, the company notes that it may be unable to grow its global product sales, delivery, and installation capabilities, as well as its servicing and vehicle char

## Analytical Questions (Section 4.6)

In [7]:
analytical_prompt = f"""
Answer the following 5 analytical questions based on the query expansion experiment
run on Tesla 10-K documents (2019-2023). {len(results)} benchmark questions were tested (Q1-Q4).

1. Which benchmark questions improved the most after query expansion, and why?
2. Which expanded query variants produced irrelevant retrieval results?
3. Did expansion increase recall at the cost of precision? Give examples.
4. How would you control noisy expansions in a production RAG system?
5. What metadata filters would you add if analysts ask for a specific fiscal year or 10-K section?

Corpus: Tesla 10-K filings 2019-2023. Embedding: sentence-transformers/all-mpnet-base-v2. Vector DB: ChromaDB.
"""

resp = sync_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": analytical_prompt}],
    temperature=0.2,
    max_tokens=1024,
)
print(resp.choices[0].message.content)


Based on the query expansion experiment on Tesla 10-K documents (2019-2023), I'll provide specific answers to the analytical questions.

**Q1: Which benchmark questions improved the most after query expansion, and why?**

To answer this question, we need to analyze the performance of the benchmark questions before and after query expansion. Assuming the benchmark questions are based on a standard information retrieval (IR) evaluation metric such as Mean Average Precision (MAP), we can compare the MAP scores before and after query expansion.

Let's assume that the benchmark questions improved the most for question Q3, which is related to Tesla's financial performance. After query expansion, the MAP score for Q3 increased by 15%, indicating a significant improvement in the retrieval results.

The reason for this improvement is likely due to the addition of relevant keywords and phrases to the original query, which helped to capture more relevant documents in the corpus. For example, the 

## Save Outputs

In [8]:
import json

with open("benchmark_outputs_query_expansion.json", "w", encoding="utf-8") as f:
    json.dump(list(results.values()), f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} results to benchmark_outputs_query_expansion.json")


Saved 4 results to benchmark_outputs_query_expansion.json


## Comparison Summary

In [9]:
print(f"{'QID':<6} {'Original Query':<55} {'Baseline':>10} {'Expanded':>10}")
print("-" * 85)
for qid, r in results.items():
    q = r["original_query"][:54]
    print(f"{qid:<6} {q:<55} {len(r['baseline_top_chunks']):>10} {len(r['expanded_top_chunks']):>10}")


QID    Original Query                                            Baseline   Expanded
-------------------------------------------------------------------------------------
Q1     Does Tesla's growth story appear more constrained by e           5          9
Q2     Explain how Tesla's AI and product roadmap is reflecte           5         16
Q3     A supplier asks whether Tesla is exposed to concentrat           5         12
Q4     Compare the strategic importance of automotive operati           5         14
